# AutoDock Tutorial
### Notebook 01. PDB Structure Preparation 

**Version 1.0.0 - April, 2026. Monterrey**


**Authors:** 
[Ana C. Murrieta ](https://orcid.org/0000-0002-7619-8880) and [Flavio F. Contreras-Torres](https://orcid.org/0000-0003-2375-131X). Tecnológico de Monterrey.



[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NanoBiostructuresRG/AutodockTutorial/blob/main/notebooks/pdb_structure_preparation.ipynb)

---


### Contents

This worksheet consists of a guided, step-by-step introduction to **protein structure inspection and preparation** using data retrieved from the **Protein Data Bank (PDB)**.

The activities are structured to be completed in approximately **30 to 60 minutes**. However, this is your personal workspace—we encourage you to add your own code cells, inspect additional PDB entries, and test alternative structural decisions as you build confidence with the workflow. The more you explore, the better prepared you'll be for the receptor preparation and docking steps that follow.

> **Note:** This notebook assumes [Biopython](https://biopython.org/) is already installed in the active environment.

---

## How to work
1. **Open the notebook**: Click on the "Open in Colab" button or use the link provided to open the tutorial in **Google Colab**.
2. **Create your workspace**: Once open, go to **File > Save a copy in Drive**. This is vital! You must work on your own copy to ensure your progress is saved.
    - **Tip**: Look at the top-left corner. If you see "Copy of...", you are ready to go!
3. **Solve the exercises**: Complete the missing parts of the code where indicated. You can run each cell to test your progress (Shift + Enter).
4. When you finish:
    - **Rename** your file following the convention:
        - `YourID_pdb_structure_preparation.ipynb`
    - **Download the file**: Go to File > Download > Download .ipynb.
    - **Upload** the downloaded file to the **CANVAS assignment module**.

**Do not** modify the notebook structure or function signatures unless explicitly stated.


---

## Protein Data Bank (PDB)

Programmatic access to structural biology databases is an efficient and automatable way to retrieve specific, relevant information for molecular modeling and bioinformatics tasks. Among the available repositories, the [Protein Data Bank (PDB)](https://www.rcsb.org/) is one of the most widely used by the scientific community. It provides access to experimentally determined three-dimensional structures of biological macromolecules, including proteins, nucleic acids, and their complexes with ligands, cofactors, and other biomolecules.

**The Protein Data Bank (PDB)** is a public structural biology resource that archives atomic-level 3D structures of biomolecules, generated through techniques such as X-ray crystallography, NMR spectroscopy, and cryo-electron microscopy.

In this notebook, we will **search the PDB**, **retrieve structural information programmatically**, and **prepare selected biomolecular structures** for downstream analysis in molecular docking and related computational workflows.

---


### Step 0: The Computational Environment

Before importing biomolecular structures, we must prepare a minimal computational environment for communicating with the **PDB** and storing retrieved files locally.

First, we rely on a lightweight Python stack for web access and file handling:

* **`requests`**: A widely used library for making HTTP requests. It allows our script to connect to the RCSB Protein Data Bank and download structural files directly from the archive.
* **`pathlib`**: A modern and convenient library for file system management. It helps us create folders and define file paths in a clean, platform-independent way.
* **`Bio.PDB`** from **Biopython**: to read and inspect macromolecular structures stored in PDBx/mmCIF format.


This first step is intentionally simple: our objective is to **establish programmatic access to PDB structures** before moving into parsing, inspection, and structure preparation workflows.

> **Note:** In structural bioinformatics, the legacy `.pdb` format is still common in many workflows. However, **RCSB recommends transitioning to the PDBx/mmCIF format**, since it is the modern archival standard and is better suited for current and future PDB entries, including upcoming extended identifiers that will not be fully supported by the classic PDB format.

In [ ]:
# Step 0
import requests
from pathlib import Path

# PDB entry ID
pdb_id = "5YCP"

# Create output directory
project_root = Path.cwd().parent
cif_dir = project_root / "docking" / "cifs"
cif_dir.mkdir(parents=True, exist_ok=True)

# Recommended modern format: mmCIF
url = f"https://files.rcsb.org/download/{pdb_id}.cif"
output_file = cif_dir / f"{pdb_id}.cif"

# Download structure
response = requests.get(url)
response.raise_for_status()

with open(output_file, "wb") as f:
    f.write(response.content)

print(f"Structure downloaded successfully: {output_file}")

In [ ]:
print("Project root:", project_root)
print("CIF directory:", cif_dir)
print("Downloaded file:", output_file)

### Step 1: Load and Inspection

This step is aimed to verify that the downloaded file can be read correctly and to inspect its basic structural organization. At this stage, we are not yet modifying the biomolecule. Instead, we focus on understanding what the structure contains, such as:

* the number of models,
* the available chains,
* the residue composition,
* and the presence of heteroatoms such as ligands, cofactors, or crystallographic waters.

This inspection step is essential before structure preparation, since it reveals the basic organization of the structure and helps distinguish the main biomolecular target from smaller structural elements and solvent molecules. In docking-oriented preparation workflows, such differences guide later decisions about which parts of the biomolecule should be preserved in subsequent preparation steps, which chains to retain, which waters to remove, and whether any hetero components should be preserved for biological relevance.



In [ ]:
# Step 1
from pathlib import Path
from Bio.PDB import MMCIFParser

pdb_id = "5YCP"

project_root = Path.cwd().parent
cif_dir = project_root / "docking" / "cifs"
cif_file = cif_dir / f"{pdb_id}.cif"

parser = MMCIFParser(QUIET=True)
structure = parser.get_structure(pdb_id, cif_file)

print(f"Structure loaded successfully: {pdb_id}")
print(f"Models found: {len(list(structure))}")

for model in structure:
    print(f"\nModel {model.id}")
    print("Chains:", [chain.id for chain in model])

The output confirms that the **5YCP** structure was read successfully from the downloaded mmCIF file. 

- The line **`Models found: 1`** indicates that the entry contains a single structural model. This is the most common case for structures determined by techniques such as X-ray crystallography.
- The line **`Model 0`** refers to the identifier assigned by Biopython to the first model in the structure. Since Python uses zero-based indexing, the first and only model is labeled as **0**.
- Finally, **`Chains: ['A', 'B']`** shows that this structure contains two chains, named **A** and **B**. In structural biology, chains usually represent individual polypeptide subunits, nucleic acid strands, or other distinct biomolecular components within the same deposited entry.


### Step 2: Inspect Chain and Residue Content

At this stage, it is useful to examine each chain separately and estimate how many residues are present, as well as whether the structure contains non-standard components such as ligands, cofactors, ions, or crystallographic water molecules.

This information is essential for structure preparation because not every component in the deposited file should necessarily be retained for docking experiments.

In [5]:
# Step 2
for model in structure:
    print(f"\nModel {model.id}")
    
    for chain in model:
        total_residues = 0
        standard_residues = 0
        water_molecules = 0
        hetero_residues = 0
        
        for residue in chain:
            total_residues += 1
            hetflag = residue.id[0]
            
            if hetflag == " ":
                standard_residues += 1
            elif hetflag == "W":
                water_molecules += 1
            else:
                hetero_residues += 1
        
        print(f"\nChain {chain.id}")
        print(f"  Total residues: {total_residues}")
        print(f"  Standard residues: {standard_residues}")
        print(f"  Water molecules: {water_molecules}")
        print(f"  Hetero residues: {hetero_residues}")


Model 0

Chain A
  Total residues: 347
  Standard residues: 263
  Water molecules: 83
  Hetero residues: 1

Chain B
  Total residues: 13
  Standard residues: 11
  Water molecules: 2
  Hetero residues: 0


The output shows that the two chains in **5YCP** differ substantially in composition and size.

- **Chain A** is the main macromolecular component of the structure. It contains **347 total residues**, of which **263 are standard residues**. This chain likely corresponds to the principal biological polymer in the entry. In addition, it contains **83 water molecules** and **1 hetero residue**, suggesting the presence of crystallographic solvent and at least one non-standard component, such as a ligand, ion, or cofactor.

- **Chain B** is much smaller, with only **13 total residues**, including **11 standard residues** and **2 water molecules**. Since it lacks hetero residues and is far shorter than chain A, it may represent a short peptide fragment, a secondary structural partner, or another minor component of the deposited complex.



### Step 3: Identify Non-Standard Components

In this step we will identify the **non-standard components** present in the structure. Our goal is to determine exactly which hetero residues are present, where they are located, and in which chain they appear.

In PDB entries, these components may include **ligands, cofactors, metal ions, buffer molecules, crystallographic additives, or other hetero residues** that are not part of the standard polymer sequence. Although they are grouped together as hetero components, their biological relevance can vary significantly.

> **Note**: Some non-standard components may need to be preserved, while others can be removed depending on the docking objective.

In [ ]:
# Step 3
for model in structure:
    print(f"\nModel {model.id}")
    
    for chain in model:
        print(f"\nChain {chain.id}")
        found_hetero = False
        
        for residue in chain:
            hetflag, resseq, icode = residue.id
            
            if hetflag not in (" ", "W"):
                found_hetero = True
                print(
                    f"  Hetero residue found: {residue.resname} | "
                    f"Residue number: {resseq} | "
                    f"Insertion code: {icode}"
                )
        
        if not found_hetero:
            print("  No hetero residues found.")

The inspection shows that the structure contains a single non-standard residue: **BRL**, located in **chain A** as residue **501**.

**BRL** corresponds to a small organic molecule:
**2,4-thiazolidinedione, 5-[[4-[2-(methyl-2-pyridinylamino)ethoxy]phenyl]methyl]-(9CI)**, which is a **bound ligand** associated with the main biomolecular chain. 

Bound ligands are especially important in docking-oriented workflows because they often reveal the location of a biologically relevant binding site and can serve as structural references for receptor preparation, binding-site definition, and protocol validation.


### Step 4: Inspect the Bound Ligand

In this exercise, the hetero residue **BRL** corresponds to a **bound small-molecule ligand** associated with the main protein chain. Co-crystallized ligands are especially informative in docking workflows because they can reveal the location of a biologically relevant binding site and provide a structural reference for receptor preparation.

Now, we will examine the ligand in more detail by confirming its residue name, chain location, residue number, and atomic composition. This inspection will help us decide how to use the ligand in later steps, such as defining the binding site, validating the docking setup, or separating receptor and ligand into independent files.


In [ ]:
# Step 4
target_resname = "BRL"

for model in structure:
    print(f"\nModel {model.id}")
    
    for chain in model:
        for residue in chain:
            hetflag, resseq, icode = residue.id
            
            if residue.resname == target_resname and hetflag not in (" ", "W"):
                atoms = list(residue.get_atoms())
                
                print(f"\nLigand found: {residue.resname}")
                print(f"Chain: {chain.id}")
                print(f"Residue number: {resseq}")
                print(f"Insertion code: {icode}")
                print(f"Number of atoms: {len(atoms)}")
                
                print("\nAtom list:")
                for atom in atoms:
                    coord = atom.coord
                    print(
                        f"  {atom.get_name():<4} "
                        f"({coord[0]:8.3f}, {coord[1]:8.3f}, {coord[2]:8.3f})"
                    )

### Step 5: Inspect Standard Residue Composition

Although the previous summary revealed how many standard residues, waters, and hetero components are present, it does not yet clarify the structural identity of each chain. At this stage, we want to determine how the standard residues are distributed, inspect their residue numbering, and verify the general composition of each chain.

Biostructures may contain more than one chain, and because not all chains necessarily correspond to the main receptor, it is important to examine the standard residue composition of each chain in greater detail. Some residues may represent short peptide fragments, auxiliary partners, or other structural elements that may or may not be relevant for docking preparation.

In [ ]:
# Step 5
for model in structure:
    print(f"\nModel {model.id}")
    
    for chain in model:
        standard_residues = []
        
        for residue in chain:
            hetflag, resseq, icode = residue.id
            if hetflag == " ":
                standard_residues.append((residue.resname, resseq, icode))
        
        print(f"\nChain {chain.id}")
        print(f"  Number of standard residues: {len(standard_residues)}")
        
        if standard_residues:
            first_res = standard_residues[0]
            last_res = standard_residues[-1]
            
            first_icode = first_res[2].strip() if first_res[2].strip() else ""
            last_icode = last_res[2].strip() if last_res[2].strip() else ""
            
            print(f"  First residue: {first_res[0]} {first_res[1]}{first_icode}")
            print(f"  Last residue:  {last_res[0]} {last_res[1]}{last_icode}")
            
            print("  First 10 standard residues:")
            for resname, resseq, icode in standard_residues[:10]:
                ins_code = icode.strip() if icode.strip() else ""
                print(f"    {resname} {resseq}{ins_code}")

The output indicates that **chain A** contains **263 standard residues**, spanning from **LEU 204** to **TYR 477**, which is consistent with a substantial protein fragment suitable for receptor-oriented analysis. 

At first glance, the residue numbering may seem inconsistent with the residue count. Since the chain starts at **204** and ends at **477**, a continuous numbering scheme would suggest more residues than those actually observed $(477 - 204 + 1 = 274)$. This indicates that the numbering is **not fully continuous** in the resolved structure. However, such discrepancies are common and usually in experimental structural files because they reflect **missing or unresolved residues**, flexible loop regions not modeled in the electron density, or numbering inherited from the original biological construct.

Moreover, the residue numbering begins at **204** rather than **1**, indicating that the deposited structure likely represents a specific region or construct derived from a larger protein sequence.

In contrast, **chain B** contains only **11 standard residues**, ranging from **ARG 686** to **GLU 696**. Its small size suggests that it is not an independent receptor chain, but rather a short peptide or auxiliary structural component.

The next cell will help us to identify where are the numbering gaps.


In [ ]:
# Detect numbering gaps between standard residues in each chain
for model in structure:
    print(f"\nModel {model.id}")
    
    for chain in model:
        standard_numbers = []
        
        for residue in chain:
            hetflag, resseq, icode = residue.id
            if hetflag == " ":
                standard_numbers.append((residue.resname, resseq, icode))
        
        print(f"\nChain {chain.id}")
        
        if len(standard_numbers) < 2:
            print("  Not enough standard residues to evaluate gaps.")
            continue
        
        gaps_found = False
        
        for i in range(len(standard_numbers) - 1):
            current_res = standard_numbers[i]
            next_res = standard_numbers[i + 1]
            
            current_num = current_res[1]
            next_num = next_res[1]
            
            if next_num - current_num > 1:
                gaps_found = True
                print(
                    f"  Gap detected: {current_res[0]} {current_num} -> "
                    f"{next_res[0]} {next_num}"
                )
        
        if not gaps_found:
            print("  No numbering gaps detected.")

The output shows that **chain A** contains a numbering gap between **LYS 261** and **GLN 273**. This means that residues **262–272** are not present in the modeled structure.

Such gaps are common in experimental structures and usually indicate **unresolved regions**, often due to structural flexibility or insufficient experimental density.

In contrast, **chain B** shows **no numbering gaps**, meaning that its residue numbering is continuous across the resolved segment.



### Step 6: Evaluate Ligand–Chain Proximity

Here, we will examine the spatial relationship between the bound ligand **BRL** and the surrounding chains. This will help us determine whether **chain A** alone is sufficient as the receptor, or whether **chain B** should also be considered during structure preparation.

Some chains may form part of the biologically relevant binding environment, whereas others may be distant, accessory, or unrelated to the ligand-binding site.



In [ ]:
# Step 6
import numpy as np

# Find ligand BRL
ligand_atoms = []

for model in structure:
    for chain in model:
        for residue in chain:
            hetflag, resseq, icode = residue.id
            if residue.resname == "BRL" and hetflag not in (" ", "W"):
                ligand_atoms = list(residue.get_atoms())
                ligand_chain = chain.id
                ligand_resseq = resseq

if not ligand_atoms:
    raise ValueError("Ligand BRL was not found in the structure.")

print(f"Ligand identified: BRL | Chain {ligand_chain} | Residue {ligand_resseq}")

# Compare ligand proximity to each chain
for model in structure:
    print(f"\nModel {model.id}")
    
    for chain in model:
        chain_atoms = []
        
        for residue in chain:
            hetflag = residue.id[0]
            if hetflag == " ":  # standard residues only
                chain_atoms.extend(list(residue.get_atoms()))
        
        if not chain_atoms:
            print(f"Chain {chain.id}: no standard atoms found.")
            continue
        
        min_distance = float("inf")
        closest_lig_atom = None
        closest_chain_atom = None
        
        for latom in ligand_atoms:
            lcoord = latom.coord
            for catom in chain_atoms:
                ccoord = catom.coord
                distance = np.linalg.norm(lcoord - ccoord)
                
                if distance < min_distance:
                    min_distance = distance
                    closest_lig_atom = latom
                    closest_chain_atom = catom
        
        print(f"\nChain {chain.id}")
        print(f"  Minimum distance to BRL: {min_distance:.3f} Å")
        print(f"  Closest ligand atom: {closest_lig_atom.get_name()}")
        print(
            f"  Closest chain atom: {closest_chain_atom.get_full_id()[2]} / "
            f"{closest_chain_atom.get_parent().resname} "
            f"{closest_chain_atom.get_parent().id[1]} / "
            f"{closest_chain_atom.get_name()}"
        )

The proximity analysis shows that **BRL is closely associated with chain A**, with a minimum distance of **2.703 Å**. This strongly suggests that chain A contains the relevant ligand-binding environment and should be treated as the primary receptor chain.

In contrast, **chain B** is much farther from the ligand, with a minimum distance of **11.688 Å**. This indicates that it is not part of the immediate binding site and is unlikely to be essential for an initial docking-oriented receptor model.

Based on this result, **chain A** can be selected as the main receptor component for subsequent structure preparation, while **chain B** may be excluded in a simplified receptor setup centered on the bound ligand.

### Step 7: Inspect Water Molecules Near the Bound Ligand

In this step, we will examine which water molecules are located near **BRL** and estimate their distances to the ligand. This analysis helps distinguish bulk solvent from waters that may be relevant for the binding environment and should therefore be considered more carefully during preparation.

Although many water molecules in a deposited structure correspond to solvent and may be excluded during receptor preparation, some can occupy positions close to the binding site and potentially contribute to ligand recognition or local structural stabilization.

In [ ]:
# Step 7
import numpy as np

# Define ligand and distance cutoff
target_resname = "BRL"
distance_cutoff = 3.5  # Å

# Locate ligand atoms
ligand_atoms = []

for model in structure:
    for chain in model:
        for residue in chain:
            hetflag, resseq, icode = residue.id
            if residue.resname == target_resname and hetflag not in (" ", "W"):
                ligand_atoms = list(residue.get_atoms())

if not ligand_atoms:
    raise ValueError(f"Ligand {target_resname} was not found in the structure.")

# Search for nearby water molecules and store atom-level closest contact
nearby_waters = []

for model in structure:
    for chain in model:
        for residue in chain:
            hetflag, resseq, icode = residue.id

            if hetflag == "W":  # water molecule
                water_atoms = list(residue.get_atoms())
                min_distance = float("inf")
                closest_water_atom = None
                closest_ligand_atom = None

                for watom in water_atoms:
                    for latom in ligand_atoms:
                        distance = np.linalg.norm(watom.coord - latom.coord)
                        if distance < min_distance:
                            min_distance = distance
                            closest_water_atom = watom
                            closest_ligand_atom = latom

                if min_distance <= distance_cutoff:
                    nearby_waters.append({
                        "chain": chain.id,
                        "water_resname": residue.resname,
                        "water_resseq": residue.id[1],
                        "water_atom": closest_water_atom.get_name(),
                        "ligand_atom": closest_ligand_atom.get_name(),
                        "distance": min_distance
                    })

# Report results
print(f"Water molecules within {distance_cutoff:.1f} Å of {target_resname}:")

if nearby_waters:
    for water in sorted(nearby_waters, key=lambda x: x["distance"]):
        print(
            f"  Chain {water['chain']} | {water['water_resname']} {water['water_resseq']} | "
            f"Water atom: {water['water_atom']} | "
            f"Closest ligand atom: {water['ligand_atom']} | "
            f"Minimum distance: {water['distance']:.3f} Å"
        )
else:
    print("  No nearby water molecules found.")

The closest water molecule to the bound ligand is **HOH 642** in **chain A**. Its oxygen atom is located **2.822 Å** from ligand atom **N18**.

This short distance places the water molecule within the immediate binding environment and suggests that it may participate in a local polar interaction network around the ligand. Although distance alone is not sufficient to confirm a hydrogen bond, the contact is close enough to justify keeping this water molecule under consideration during structure preparation.

### Step 8: Separate Receptor and Ligand Components

At this stage, the main structural elements of the complex have been characterized sufficiently to define a receptor preparation strategy.

In docking-oriented workflows, it is often necessary to treat the **receptor** and the **ligand** as independent molecular entities. The receptor usually corresponds to the macromolecular target, while the ligand might be extracted as a separate molecule, and it may serve as a reference for defining the binding site, validating the docking protocol, or preparing a redocking experiment.

The analysis indicates that **chain A** contains the bound ligand environment and represents the primary receptor component, whereas **chain B** is distant from the ligand and is unlikely to be essential for an initial docking-oriented receptor model. In addition, most crystallographic waters appear to correspond to bulk solvent, but **HOH 642** is positioned close to the ligand and may contribute to the local binding environment.

Based on these observations, the initial receptor model will be defined using **chain A** as the core receptor component. Two preparation variants can then be considered: one in which **HOH 642** is removed together with the remaining crystallographic waters, and another in which **HOH 642** is retained as a potentially relevant binding-site water. Generating both versions allows the effect of this water molecule to be evaluated explicitly in later docking steps.

Here, we will isolate the protein component from the bound ligand and generate independent structure files for each one.



In [ ]:
# Step 8
from Bio.PDB import PDBIO, Select

# User-defined structure components
receptor_chain_id = "A"
ligand_resname = "BRL"

# Define output directory
output_dir = project_root / "docking" / "prepared"
output_dir.mkdir(parents=True, exist_ok=True)

# Define output file names
receptor_file = output_dir / f"{pdb_id}_chain{receptor_chain_id}_receptor.pdb"
ligand_file = output_dir / f"{pdb_id}_{ligand_resname}_ligand.pdb"


# Select only standard residues from the chosen receptor chain
class ReceptorSelect(Select):
    def accept_chain(self, chain):
        return chain.id == receptor_chain_id
    
    def accept_residue(self, residue):
        return residue.id[0] == " "


# Select only the ligand defined by ligand_resname
class LigandSelect(Select):
    def accept_residue(self, residue):
        hetflag, resseq, icode = residue.id
        return residue.resname == ligand_resname and hetflag not in (" ", "W")


# Save receptor
io = PDBIO()
io.set_structure(structure)
io.save(str(receptor_file), ReceptorSelect())

# Save ligand
io = PDBIO()
io.set_structure(structure)
io.save(str(ligand_file), LigandSelect())

print("Files generated successfully:")
print(f"  Receptor: {receptor_file}")
print(f"  Ligand:   {ligand_file}")







### Step 9: Verify the Exported Structure Files

Finally, we want to confirm that the receptor file includes only the selected protein chain without water molecules or hetero components, and that the ligand file contains only the isolated bound ligand.

This verification step helps detect selection or export errors before proceeding to downstream preparation tasks such as protonation, charge assignment, file conversion, or docking setup.

In [ ]:
# Step 9
from Bio.PDB import PDBParser

# Define exported files
receptor_file = output_dir / f"{pdb_id}_chain{receptor_chain_id}_receptor.pdb"
ligand_file = output_dir / f"{pdb_id}_{ligand_resname}_ligand.pdb"

# Initialize parser
parser = PDBParser(QUIET=True)

# Load exported structures
receptor_structure = parser.get_structure("receptor", receptor_file)
ligand_structure = parser.get_structure("ligand", ligand_file)


def summarize_structure(structure, label):
    print(f"\n{label}")
    
    for model in structure:
        print(f"  Model {model.id}")
        
        for chain in model:
            total_residues = 0
            standard_residues = 0
            water_molecules = 0
            hetero_residues = 0
            atom_count = 0
            
            for residue in chain:
                total_residues += 1
                atom_count += len(list(residue.get_atoms()))
                
                hetflag = residue.id[0]
                if hetflag == " ":
                    standard_residues += 1
                elif hetflag == "W":
                    water_molecules += 1
                else:
                    hetero_residues += 1
            
            print(f"    Chain {chain.id}")
            print(f"      Total residues: {total_residues}")
            print(f"      Standard residues: {standard_residues}")
            print(f"      Water molecules: {water_molecules}")
            print(f"      Hetero residues: {hetero_residues}")
            print(f"      Total atoms: {atom_count}")


# Print summaries
summarize_structure(receptor_structure, "Receptor summary")
summarize_structure(ligand_structure, "Ligand summary")

### Summary and Next Step

You have successfully retrieved, inspected, and separated the main structural components of a protein–ligand complex from the Protein Data Bank.

**Current State:**  
We now have:

- the original **PDBx/mmCIF** structure downloaded from the **RCSB Protein Data Bank**
- a validated identification of the main receptor component (**chain A**)
- the co-crystallized ligand identified and inspected
- the residue composition and numbering gaps evaluated
- the spatial relationship between ligand, chains, and nearby water molecules examined
- two exported structure files ready for further preparation:
  - `PROTEIN_chainA_receptor.pdb`
  - `PROTEIN_BRL_ligand.pdb`

**Next Step:**  
In the next notebook, we will rebuild the receptor structure by using the **canonical protein sequence from UniProt** as reference. We will first retrieve the full canonical sequence, align it with the experimentally resolved structure, identify the missing segment(s), and generate an alignment file suitable for comparative modeling. This will allow us to reconstruct unresolved regions and obtain a more complete receptor model for downstream docking studies.


---

### Exercises

Now that you have completed the structural inspection and component separation workflow, use the same pipeline to analyze additional protein–ligand complexes from the Protein Data Bank.

These exercises are designed to help you practice how to inspect structural organization, identify relevant binding-site components, and decide which elements should be retained for docking-oriented receptor preparation.

> **Important:** Different PDB entries may vary substantially in structural complexity.  
> Some structures may contain multiple protein chains, several ligands, metal ions, missing regions, or large numbers of crystallographic water molecules.  
> For that reason, always inspect the structure carefully before deciding which components to keep or remove.

---

**Exercise 1: Repeat the workflow with a different PDB entry**  
Change the `pdb_id` variable and repeat the full inspection workflow with: `"6MD4"`, `"2POB"`, `"6K0T"`.

For your selected structure, determine:

1. how many models are present,
2. which chains are included,
3. which chain appears to correspond to the main receptor,
4. whether a co-crystallized ligand is present,
5. and whether any numbering gaps are detected in the receptor chain.

Summarize your observations in a short paragraph.

---

**Exercise 2: Identify the binding-site components**  
Using your new PDB entry, identify the main non-standard components in the structure.

For the ligand you select, determine:

1. its residue name,
2. the chain in which it appears,
3. the number of atoms it contains,
4. the chain that is closest to the ligand,
5. and whether any crystallographic water molecules are located within **3.5 Å** of the ligand.

Based on your results, explain which structural components you would keep for receptor preparation and why.

---

**Exercise 3: Export a new receptor–ligand pair**  
Modify the export step so that it matches the structural decisions you made for your selected PDB entry.

Generate:

- a **receptor file** containing the chain chosen for docking preparation,
- and a **ligand file** containing the selected co-crystallized ligand.

Use clear file names that include the PDB ID, receptor chain, and ligand name.  
Then verify the exported files and confirm that:

- the receptor contains only the intended protein residues,
- the ligand file contains only the intended hetero residue,
- and no unintended water molecules or extra components remain.

---


### Outlook

Congratulations! You have completed a professional workflow for retrieving, inspecting, and preparing biomolecular structures from the **Protein Data Bank (PDB)**. You now possess the skills to:

* retrieve structural files programmatically from the **RCSB Protein Data Bank**
* inspect the internal organization of a protein structure at the level of models, chains, residues, ligands, and water molecules
* identify missing regions and evaluate the structural relevance of non-standard components
* separate receptor and ligand components into clean structure files for downstream modeling and docking preparation

See you in the next lesson!

---

### License  
The content of this tutorial itself is licensed under the terms and conditions of the [Creative Commons Attribution (CC BY 4.0) license](https://creativecommons.org/licenses/by/4.0/legalcode.en), and the underlying source code used to format and display that content is licensed under the [MIT license](https://github.com/NanoBiostructuresRG/AutodockTutorial/blob/main/LICENSE). See the LICENSE files for full details.

### Attribution
If you use or adapt this material, please provide appropriate credit to the original authors, [ACM](https://orcid.org/0000-0002-7619-8880) and [FFCT](https://orcid.org/0000-0003-2375-131X), as well as to the repository: [https://github.com/NanoBiostructuresRG](https://github.com/NanoBiostructuresRG).